## 0.Title

# Video Platform User Behavior Analysis

This project analyzes user event data from a video platform to understand engagement patterns, detect anomalies, and study temporal behavior.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Data

In [9]:
df = pd.read_excel("data/event_list.xlsx")
df.head()

,account_id,server_time,screen_type,action_id,device_type,user_browser
0,102301,2021-10-01 06:58:16,main,screenview,mobile_web,Mobile Safari
1,102301,2021-10-01 06:58:30,main,screenview,mobile_web,Mobile Safari
2,102313,2021-10-01 03:40:19,main,screenview,mobile_web,Chrome
3,102313,2021-10-01 06:51:14,main,screenview,desktop_web,Chrome
4,102313,2021-10-01 06:51:17,main,click,desktop_web,Chrome


## 3. Data Understanding

In [ ]:
df.info()
df.describe()
df.isna().sum()

## 4. Filter Data

In [ ]:
df = df[df['screen_type'] == 'player']
df.head()

## 5. Task 1 - User activity analysis

### Top 10 users

In [ ]:
top_users = (
    df.groupby('account_id')
    .agg(total_actions=('action_id', 'count'))
    .reset_index()
    .sort_values('total_actions', ascending=False)
    .head(10)
)

top_users

### Visualization

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(top_users['account_id'].astype(str), top_users['total_actions'])
plt.title('Top 10 Active Users')
plt.xlabel('User ID')
plt.ylabel('Number of actions')
plt.xticks(rotation=45)
plt.show()

### Browser analysis

In [ ]:
top_user_ids = top_users['account_id']

browser_stats = (
    df[df['account_id'].isin(top_user_ids)]
    .groupby('user_browser')
    .agg(actions=('action_id', 'count'))
    .reset_index()
    .sort_values('actions', ascending=False)
)

browser_stats

### Action distribution

In [ ]:
action_stats = (
    df[df['account_id'].isin(top_user_ids)]
    .groupby('action_id')
    .agg(count=('account_id', 'count'))
    .reset_index()
    .sort_values('count', ascending=False)
)

action_stats

## 6. Task 2 - Consistency checks

### Pause without start

In [ ]:
paused_users = set(df[df['action_id'] == 'playback_pause']['account_id'])
started_users = set(df[df['action_id'] == 'playback_start']['account_id'])

pause_without_start = paused_users - started_users

len(pause_without_start)

### Start without pause

In [ ]:
start_without_pause = started_users - paused_users

len(start_without_pause)

### Interpretation
Some users demonstrate inconsistent event behavior, 
which may indicate tracking issues or non-standard usage patterns.

## 7. Task 3 - Time analysis

### Extract hour

In [ ]:
df['server_time'] = pd.to_datetime(df['server_time'])
df['hour'] = df['server_time'].dt.hour

### Hourly activity

In [ ]:
hourly = (
    df.groupby('hour')
    .agg(actions=('action_id', 'count'))
    .reset_index()
)

### Plot

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(hourly['hour'], hourly['actions'])
plt.title('User activity by hour')
plt.xlabel('Hour')
plt.ylabel('Actions')
plt.grid()
plt.show()

### Interpretation
User activity shows clear daily patterns with peak hours,
indicating structured usage behavior across the day.

## 8. Final conclucions
### Key insights

- User activity is highly uneven across accounts  
- A small group of users generates most interactions  
- Users exhibit clear daily usage patterns  
- Some inconsistencies in event sequences may indicate tracking issues  .